# 17 — Regularization, Initialization, and Stable Training

In the previous notebook, we trained a complete CNN for image classification.

Now we will study three ideas that strongly affect whether a neural network trains well and generalizes well:

1. **Regularization**
2. **Weight initialization**
3. **Training stability**

A model can have the correct architecture and still perform poorly because:

- It overfits
- It underfits
- Its weights start at a bad scale
- Activations become extremely small or large
- Gradients vanish or explode
- The learning rate is unstable
- Regularization is too weak or too strong

## In this notebook, we will study:

1. What is regularization?
2. Overfitting vs underfitting
3. Dropout in depth
4. Weight decay
5. L1 vs L2 intuition
6. Data augmentation as regularization
7. Early stopping
8. Weight initialization
9. Xavier initialization
10. Kaiming initialization
11. Why initialization matters
12. Vanishing and exploding activations
13. Vanishing and exploding gradients
14. Gradient clipping
15. Stable training habits
16. Comparing regularization strategies
17. Common mistakes
18. Debugging unstable training
19. Practice exercises

## Main Goal

By the end of this notebook, you should understand that successful training is not only:

$$
\boxed{
\text{Model}
+
\text{Loss}
+
\text{Optimizer}
}
$$

It is also about controlling:

$$
\boxed{
\text{Capacity}
+
\text{Regularization}
+
\text{Initialization}
+
\text{Gradient Scale}
+
\text{Training Dynamics}
}
$$


In [ ]:
import copy
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

print("PyTorch version:", torch.__version__)


# 1. What Is Regularization?

Regularization means adding techniques that encourage a model to generalize better to unseen data.

The goal is not simply:

> Make training loss as small as possible.

The real goal is:

> Learn patterns that continue to work on validation and test data.

A highly flexible model may memorize training-specific details.

Regularization tries to reduce this tendency.


# 2. Training Performance vs Generalization

Suppose Model A has:

$$
training\ accuracy=99\%
$$

but:

$$
validation\ accuracy=70\%
$$

Model B has:

$$
training\ accuracy=92\%
$$

and:

$$
validation\ accuracy=90\%
$$

Model B is usually the better model because it generalizes better.

The highest training accuracy is not automatically the goal.


# 3. Underfitting

Underfitting means the model does not learn the training pattern sufficiently well.

Common signs:

- Training loss remains high
- Training accuracy remains low
- Validation performance is also poor

Possible causes:

- Model too simple
- Training too short
- Learning rate poorly chosen
- Features not informative
- Excessive regularization


# 4. Overfitting

Overfitting means the model fits the training data very well but fails to generalize.

Common signs:

- Training loss becomes very small
- Training accuracy becomes very high
- Validation loss stops improving
- Validation loss may increase
- Train/validation performance gap grows


# 5. Visualizing the Three Regimes

A useful conceptual picture is:

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Regime} & \textbf{Training Performance} & \textbf{Validation Performance} \\
\hline
Underfitting & Poor & Poor \\
\hline
Good\ fit & Good & Good \\
\hline
Overfitting & Excellent & Worse \\
\hline
\end{array}
$$

The ideal model is not necessarily the one with the lowest training loss.


# 6. Common Regularization Techniques

Important regularization techniques include:

- Dropout
- Weight decay
- L1 penalties
- Data augmentation
- Early stopping
- Smaller model capacity
- More training data

Regularization techniques attack overfitting in different ways.


# 7. Dropout

Dropout randomly sets some activations to zero during training.

Example:

```python
nn.Dropout(
    p=0.5
)
```

means each activation is dropped independently with probability:

$$
p=0.5
$$

during training.


# 8. Dropout Intuition

Suppose a hidden representation is:

$$
\begin{array}{|c|c|c|c|c|}
\hline
2.0 & 1.0 & 3.0 & 4.0 & 5.0 \\
\hline
\end{array}
$$

A possible dropout mask may produce something conceptually like:

$$
\begin{array}{|c|c|c|c|c|}
\hline
0 & * & 0 & * & * \\
\hline
\end{array}
$$

where some units are removed for that training pass.

The exact mask changes randomly.


# 9. Dropout in PyTorch


In [ ]:
torch.manual_seed(42)

dropout = nn.Dropout(
    p=0.5
)

x = torch.ones(
    12
)

dropout.train()

training_output = dropout(
    x
)

print(
    "Training output:"
)

print(
    training_output
)


Notice that surviving activations may be scaled.

PyTorch uses **inverted dropout**.

During training, surviving activations are scaled so that the expected activation remains approximately unchanged.

This means evaluation does not need an extra manual scaling step.


# 10. Dropout During Evaluation

Dropout is disabled in evaluation mode.


In [ ]:
dropout.eval()

evaluation_output = dropout(
    x
)

print(
    "Evaluation output:"
)

print(
    evaluation_output
)


This is one reason why:

```python
model.train()
```

and:

```python
model.eval()
```

are essential.

Dropout behavior depends on model mode.


# 11. Comparing Dropout Modes


In [ ]:
torch.manual_seed(1)

dropout = nn.Dropout(
    p=0.5
)

x = torch.ones(
    8
)

dropout.train()

for run in range(3):
    print(
        f"Training run {run + 1}:",
        dropout(x)
    )

dropout.eval()

print(
    "Evaluation:",
    dropout(x)
)


# 12. What Does Dropout Try to Prevent?

Without dropout, hidden units can become heavily dependent on specific other units.

Dropout forces the network to repeatedly operate with different subsets of activations.

This can discourage fragile co-adaptation and improve generalization in some settings.

Dropout is not guaranteed to improve every model.


# 13. Choosing the Dropout Probability

Common values include:

$$
p=0.1
$$

$$
p=0.2
$$

$$
p=0.5
$$

But there is no universal best value.

If dropout is too weak:

- It may have little regularization effect

If dropout is too strong:

- The model may underfit
- Optimization may become harder

Tune it using validation performance.


# 14. Where Is Dropout Commonly Used?

Dropout is often used:

- Between fully connected layers
- In MLP hidden layers
- In some CNN classifier heads
- In Transformer architectures

Its placement depends on architecture.

Do not automatically insert dropout after every operation.


# 15. Simple MLP With Dropout


In [ ]:
class DropoutMLP(nn.Module):
    def __init__(
        self,
        input_dim=20,
        hidden_dim=64,
        num_classes=3,
        dropout_p=0.3
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(
                p=dropout_p
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(
                p=dropout_p
            ),

            nn.Linear(
                hidden_dim,
                num_classes
            )
        )

    def forward(self, x):
        return self.network(
            x
        )

dropout_model = DropoutMLP()

print(
    dropout_model
)


# 16. Weight Decay

Weight decay discourages model parameters from becoming unnecessarily large.

In many optimizer configurations, you specify:

```python
weight_decay=...
```

Example:

```python
torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)
```


# 17. L2 Regularization Intuition

A classical L2 penalty adds:

$$
\boxed{
\lambda
\sum_i w_i^2
}
$$

to the training objective.

So instead of optimizing only:

$$
L_{data}
$$

we optimize:

$$
\boxed{
L_{total}
=
L_{data}
+
\lambda
\sum_i w_i^2
}
$$

where:

$$
\lambda
$$

controls regularization strength.


# 18. What L2 Encourages

L2 regularization tends to encourage weights to remain smaller in magnitude.

It does not normally force many weights to exactly zero.

Conceptually, it prefers parameter configurations that solve the task without unnecessarily large values.


# 19. Manual L2 Penalty

Let's calculate a simple L2 penalty ourselves.


In [ ]:
weights = torch.tensor([
    2.0,
    -3.0,
    0.5
])

l2_penalty = (
    weights ** 2
).sum()

print(
    "L2 penalty:",
    l2_penalty.item()
)


# 20. Manual L2 Added to a Loss


In [ ]:
data_loss = torch.tensor(
    0.8
)

lambda_l2 = 0.01

total_loss = (
    data_loss
    + lambda_l2
    * l2_penalty
)

print(
    "Data loss:",
    data_loss.item()
)

print(
    "Total loss:",
    total_loss.item()
)


# 21. Weight Decay in SGD

For standard SGD, optimizer weight decay closely corresponds to classical L2-style regularization.


In [ ]:
model = nn.Linear(
    10,
    1
)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

print(
    optimizer
)


# 22. Adam vs AdamW Weight Decay

This distinction is important.

`Adam` can accept a `weight_decay` argument.

However, adaptive optimizers interact with traditional L2-style penalties differently from plain SGD.

PyTorch also provides:

`AdamW`

which uses **decoupled weight decay**.

For many modern Adam-style training setups, `AdamW` is a common choice.


In [ ]:
model = nn.Linear(
    10,
    1
)

adamw = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    adamw
)


# 23. Weight Decay Strength

Possible values might include:

$$
0
$$

$$
10^{-5}
$$

$$
10^{-4}
$$

$$
10^{-3}
$$

or larger depending on the task.

Too little regularization may not help.

Too much weight decay can cause underfitting.

Tune using validation performance.


# 24. L1 Regularization

L1 regularization adds:

$$
\boxed{
\lambda
\sum_i |w_i|
}
$$

to the objective.

So:

$$
L_{total}
=
L_{data}
+
\lambda
\sum_i |w_i|
$$


# 25. L1 vs L2 Intuition

$$
\begin{array}{|c|c|}
\hline
\textbf{L1} & \textbf{L2} \\
\hline
\sum |w_i| & \sum w_i^2 \\
\hline
Can\ encourage\ sparsity & Encourages\ smaller\ weights \\
\hline
Nonsmooth\ at\ zero & Smooth\ around\ zero \\
\hline
\end{array}
$$

L1 can push some parameters toward exactly or nearly zero more strongly than L2.


In [ ]:
weights = torch.tensor([
    2.0,
    -3.0,
    0.5
])

l1_penalty = torch.abs(
    weights
).sum()

l2_penalty = (
    weights ** 2
).sum()

print(
    "L1:",
    l1_penalty.item()
)

print(
    "L2:",
    l2_penalty.item()
)


# 26. Manual L1 Penalty for a Model

PyTorch optimizers do not generally expose an `l1_regularization=` argument.

If you want a simple L1 penalty, you can add it manually to the loss.


In [ ]:
model = nn.Linear(
    4,
    2
)

l1_strength = 1e-4

l1_penalty = torch.tensor(
    0.0
)

for parameter in model.parameters():
    l1_penalty = (
        l1_penalty
        + parameter.abs().sum()
    )

print(
    "L1 penalty:",
    l1_penalty.item()
)


# 27. Data Augmentation as Regularization

Data augmentation modifies training examples while preserving their labels.

This effectively exposes the model to more input variation.

For images, examples can include:

- Small rotations
- Crops
- Translations
- Flips
- Noise
- Intensity changes

This reduces the model's ability to simply memorize exact training pixels.


# 28. Why Augmentation Is a Form of Regularization

Suppose the model sees only one fixed training image.

It can potentially memorize details of that exact image.

If the same semantic example appears with many realistic variations, the model must learn more robust features.

So augmentation regularizes by modifying the **data distribution seen during training**.


# 29. Augmentation Must Preserve the Label

This is critical.

An augmentation is useful only when:

$$
\boxed{
label(x)
=
label(augmentation(x))
}
$$

for the task.

For medical imaging, transformations must be anatomically and clinically plausible.

A transformation that changes label meaning is not regularization — it is corrupted supervision.


# 30. Early Stopping as Regularization

Early stopping monitors validation performance.

If validation performance stops improving, training is stopped.

This prevents the model from continuing to fit training-specific details after generalization has peaked.


# 31. Basic Early-Stopping Logic


In [ ]:
best_val_loss = float(
    "inf"
)

epochs_without_improvement = 0

patience = 3

example_val_losses = [
    0.90,
    0.72,
    0.61,
    0.58,
    0.59,
    0.60,
    0.62
]

for epoch, val_loss in enumerate(
    example_val_losses,
    start=1
):
    if val_loss < best_val_loss:
        best_val_loss = (
            val_loss
        )

        epochs_without_improvement = 0

        print(
            f"Epoch {epoch}: improved"
        )

    else:
        epochs_without_improvement += 1

        print(
            f"Epoch {epoch}: "
            f"no improvement "
            f"({epochs_without_improvement})"
        )

    if (
        epochs_without_improvement
        >= patience
    ):
        print(
            "Early stopping."
        )

        break


# 32. Early Stopping Does Not Replace Checkpointing

A good workflow is:

1. Monitor validation metric
2. Save best model state
3. Stop after patience is exceeded
4. Restore the best state

The model at the stopping epoch is not necessarily the best model.


# 33. Why Initialization Matters

Before training begins, model parameters must have initial values.

Training starts from those values.

Bad initialization can cause:

- Activations to become too small
- Activations to become too large
- Gradients to vanish
- Gradients to explode
- Optimization to become slow or unstable

Initialization should keep signal magnitudes in a useful range.


# 34. What Happens With All-Zero Weights?

Suppose all neurons in the same layer begin with identical weights.

Then they receive identical gradients and remain identical.

This is called a:

> **Symmetry problem**

For hidden layers, setting all weights to zero prevents neurons from learning different features.

Biases can often safely start at zero, but weights should generally not all start identically.


In [ ]:
layer = nn.Linear(
    4,
    3
)

with torch.no_grad():
    layer.weight.zero_()
    layer.bias.zero_()

print(
    layer.weight
)


# 35. Random Initialization Breaks Symmetry

Randomly initialized weights cause different neurons to begin with different functions.

Then gradient descent can make them specialize differently.

But the **scale** of the random initialization matters.

Too small:

$$
activations\rightarrow0
$$

Too large:

$$
activations\rightarrow huge
$$


# 36. PyTorch Default Initialization

PyTorch layers such as:

- `nn.Linear`
- `nn.Conv2d`

already use sensible default initialization schemes.

So beginners do not need to manually initialize every model.

However, understanding initialization is important for:

- Custom architectures
- Research experiments
- Debugging
- Reproducing published models


In [ ]:
torch.manual_seed(42)

layer = nn.Linear(
    100,
    50
)

print(
    "Weight mean:",
    layer.weight.mean().item()
)

print(
    "Weight std:",
    layer.weight.std().item()
)


# 37. Fan-In and Fan-Out

Initialization formulas often depend on:

- `fan_in`
- `fan_out`

For a linear layer:

$$
fan\_in=in\_features
$$

$$
fan\_out=out\_features
$$

For a convolution, they also depend on kernel size and channel counts.

These quantities describe how many connections enter or leave a unit.


# 38. Xavier / Glorot Initialization

Xavier initialization is designed to help preserve signal variance through layers.

It is commonly associated with activations such as:

- Tanh
- Sigmoid-like networks
- Linear activations

PyTorch provides:

- `nn.init.xavier_uniform_`
- `nn.init.xavier_normal_`


# 39. Xavier Uniform Initialization

A common Xavier uniform form samples approximately from:

$$
\boxed{
U\left(
-\sqrt{\frac{6}{fan_{in}+fan_{out}}},
\sqrt{\frac{6}{fan_{in}+fan_{out}}}
\right)
}
$$

The exact implementation can also include a gain factor.


In [ ]:
layer = nn.Linear(
    100,
    50
)

nn.init.xavier_uniform_(
    layer.weight
)

nn.init.zeros_(
    layer.bias
)

print(
    "Weight mean:",
    layer.weight.mean().item()
)

print(
    "Weight std:",
    layer.weight.std().item()
)


# 40. Xavier Normal Initialization


In [ ]:
layer = nn.Linear(
    100,
    50
)

nn.init.xavier_normal_(
    layer.weight
)

nn.init.zeros_(
    layer.bias
)

print(
    "Weight std:",
    layer.weight.std().item()
)


# 41. Kaiming / He Initialization

Kaiming initialization is designed for rectifier-style activations such as:

> **ReLU**

PyTorch provides:

- `nn.init.kaiming_uniform_`
- `nn.init.kaiming_normal_`

For ReLU networks, Kaiming initialization is a common choice.


# 42. Kaiming Intuition

ReLU sets negative activations to zero.

That changes variance as signals move through the network.

Kaiming initialization accounts for this behavior and chooses an appropriate weight scale.

A common Kaiming-normal scale for ReLU is related to:

$$
\boxed{
\sqrt{
\frac{2}{fan_{in}}
}
}
$$


In [ ]:
layer = nn.Linear(
    100,
    50
)

nn.init.kaiming_normal_(
    layer.weight,
    nonlinearity="relu"
)

nn.init.zeros_(
    layer.bias
)

print(
    "Weight std:",
    layer.weight.std().item()
)


# 43. Xavier vs Kaiming

A useful beginner rule:

$$
\begin{array}{|c|c|}
\hline
\textbf{Activation} & \textbf{Common Initialization} \\
\hline
ReLU & Kaiming/He \\
\hline
Tanh & Xavier/Glorot \\
\hline
Linear & Xavier\ often\ reasonable \\
\hline
\end{array}
$$

This is a guideline, not an absolute law.

Architecture details can change the best choice.


# 44. Custom Initialization Function

We can initialize different layer types explicitly.


In [ ]:
def initialize_relu_network(
    module
):
    if isinstance(
        module,
        nn.Linear
    ):
        nn.init.kaiming_normal_(
            module.weight,
            nonlinearity="relu"
        )

        if module.bias is not None:
            nn.init.zeros_(
                module.bias
            )

    elif isinstance(
        module,
        nn.Conv2d
    ):
        nn.init.kaiming_normal_(
            module.weight,
            nonlinearity="relu"
        )

        if module.bias is not None:
            nn.init.zeros_(
                module.bias
            )


# 45. Applying Initialization Recursively

Every `nn.Module` supports:

```python
model.apply(function)
```

which recursively applies the function to submodules.


In [ ]:
model = nn.Sequential(
    nn.Linear(
        20,
        64
    ),
    nn.ReLU(),

    nn.Linear(
        64,
        32
    ),
    nn.ReLU(),

    nn.Linear(
        32,
        3
    )
)

model.apply(
    initialize_relu_network
)

print(
    model
)


# 46. A Nuance About the Final Layer

The final classification layer may not be followed by ReLU.

Therefore, applying ReLU-oriented initialization to every linear layer is a simplification.

In many practical models this still trains fine, but a more careful design may initialize hidden ReLU layers differently from the final output layer.

Initialization should match the actual architecture.


# 47. What Are Activations?

An activation is the output of a layer or activation function during the forward pass.

For:

$$
x
\rightarrow
Linear
\rightarrow
ReLU
$$

we may write:

$$
z=Wx+b
$$

then:

$$
a=ReLU(z)
$$

The distribution of $a$ matters as data travels through many layers.


# 48. Vanishing Activations

If each layer repeatedly reduces signal magnitude, activations can become extremely small.

After many layers:

$$
activation\ magnitude
\rightarrow0
$$

Then deeper layers may receive almost no useful signal.


# 49. Exploding Activations

If layers repeatedly amplify signal magnitude, activations can become very large.

After many layers:

$$
activation\ magnitude
\rightarrow huge
$$

This can produce:

- Numerical instability
- Huge losses
- Huge gradients
- `inf`
- `nan`


# 50. Demonstrating Activation Scale

Let's build deep linear stacks with intentionally tiny and huge weight scales.


In [ ]:
def make_scaled_deep_network(
    width=64,
    depth=12,
    weight_std=0.1
):
    layers = []

    for _ in range(depth):
        layer = nn.Linear(
            width,
            width,
            bias=False
        )

        nn.init.normal_(
            layer.weight,
            mean=0.0,
            std=weight_std
        )

        layers.append(
            layer
        )

    return nn.Sequential(
        *layers
    )


# 51. Tracking Activation Standard Deviation


In [ ]:
def activation_stds(
    model,
    x
):
    stds = []

    with torch.no_grad():
        for layer in model:
            x = layer(
                x
            )

            stds.append(
                x.std().item()
            )

    return stds

x = torch.randn(
    256,
    64
)

small_scale_model = (
    make_scaled_deep_network(
        weight_std=0.02
    )
)

large_scale_model = (
    make_scaled_deep_network(
        weight_std=0.5
    )
)

small_stds = activation_stds(
    small_scale_model,
    x
)

large_stds = activation_stds(
    large_scale_model,
    x
)

print(
    "Small-scale final std:",
    small_stds[-1]
)

print(
    "Large-scale final std:",
    large_stds[-1]
)


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    small_stds,
    marker="o",
    label="Very small initialization"
)

plt.plot(
    large_stds,
    marker="o",
    label="Very large initialization"
)

plt.xlabel("Layer")
plt.ylabel("Activation Standard Deviation")
plt.title("Activation Scale Through a Deep Network")
plt.yscale("log")
plt.legend()
plt.show()


# 52. Why This Matters

If signal scale changes dramatically across depth, optimization becomes difficult.

Good initialization tries to keep activation statistics in a useful range.

Modern architectures also use techniques such as:

- Batch Normalization
- Layer Normalization
- Residual connections

to improve signal propagation.

We will study some of these later.


# 53. Vanishing Gradients

The backward pass uses the chain rule.

For many layers, gradients involve products of many derivatives.

If those derivatives are repeatedly smaller than 1:

$$
gradient\ magnitude
\rightarrow0
$$

as we move backward.

Earlier layers may learn extremely slowly.


# 54. Exploding Gradients

If repeated derivative factors are large:

$$
gradient\ magnitude
\rightarrow huge
$$

This can cause:

- Unstable updates
- Sudden loss spikes
- `inf`
- `nan`

Gradient scale is one of the most important things to inspect when training becomes unstable.


# 55. Inspecting Gradient Norms

After:

```python
loss.backward()
```

we can inspect parameter gradient norms.


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(
        20,
        64
    ),
    nn.ReLU(),

    nn.Linear(
        64,
        64
    ),
    nn.ReLU(),

    nn.Linear(
        64,
        3
    )
)

inputs = torch.randn(
    32,
    20
)

targets = torch.randint(
    0,
    3,
    (32,)
)

criterion = nn.CrossEntropyLoss()

logits = model(
    inputs
)

loss = criterion(
    logits,
    targets
)

loss.backward()

for name, parameter in model.named_parameters():
    if parameter.grad is not None:
        print(
            name,
            "| grad norm:",
            parameter.grad.norm().item()
        )


# 56. Total Gradient Norm

We can also reason about one total gradient norm across parameters.


In [ ]:
squared_norm_sum = 0.0

for parameter in model.parameters():
    if parameter.grad is not None:
        squared_norm_sum += (
            parameter.grad.norm().item()
            ** 2
        )

total_grad_norm = math.sqrt(
    squared_norm_sum
)

print(
    "Total gradient norm:",
    total_grad_norm
)


# 57. Gradient Clipping

Gradient clipping limits gradient magnitude before the optimizer update.

PyTorch provides:

```python
torch.nn.utils.clip_grad_norm_
```

Typical sequence:

```python
optimizer.zero_grad()

loss.backward()

clip_grad_norm_(
    model.parameters(),
    max_norm=...
)

optimizer.step()
```


In [ ]:
clipped_norm = torch.nn.utils.clip_grad_norm_(
    model.parameters(),
    max_norm=1.0
)

print(
    "Norm before clipping:",
    clipped_norm.item()
)


# 58. What Gradient Clipping Does

Norm-based clipping says:

If total gradient norm is already below the threshold:

> Leave it unchanged.

If the norm is above:

$$
max\_norm
$$

scale gradients down so the norm is controlled.

It does not change the forward pass.

It changes the gradient before the optimizer step.


# 59. Gradient Clipping Is Not a Universal Fix

If gradients explode because of a deeper issue, clipping can hide symptoms.

Also inspect:

- Learning rate
- Initialization
- Input scaling
- Loss function
- Architecture
- Numerical operations

Clipping is a useful tool, not a substitute for debugging.


# 60. Gradient-Value Clipping

PyTorch also provides:

```python
torch.nn.utils.clip_grad_value_
```

This clips individual gradient values to a fixed range.

Norm clipping is more common in many deep-learning workflows.


In [ ]:
torch.nn.utils.clip_grad_value_(
    model.parameters(),
    clip_value=0.5
)

print(
    "Gradient values clipped."
)


# 61. Stable Training Habit — Normalize Inputs

Inputs with wildly different scales can make optimization harder.

For tabular data, standardization may be:

$$
x_{norm}
=
\frac{x-\mu}{\sigma}
$$

For images, use a consistent, training-derived normalization scheme.


# 62. Stable Training Habit — Start With a Reasonable Learning Rate

If loss immediately becomes:

- `nan`
- `inf`
- Extremely large

the learning rate may be too high.

A stable workflow starts with a reasonable baseline and changes one thing at a time.


# 63. Stable Training Habit — Inspect One Batch First

Before training for 100 epochs:

1. Check input shape
2. Check target shape
3. Check dtypes
4. Run one forward pass
5. Compute one loss
6. Run one backward pass
7. Check gradients
8. Verify parameters change

This catches many problems quickly.


# 64. Stable Training Habit — Monitor Train and Validation Curves

Track:

- Training loss
- Validation loss
- Training metric
- Validation metric

These curves reveal:

- Underfitting
- Overfitting
- Instability
- Poor learning rate
- Bad checkpoint timing


# 65. Stable Training Habit — Watch for Non-Finite Values

Use:

```python
torch.isfinite(...)
```

to detect:

- `nan`
- `inf`


In [ ]:
example_loss = torch.tensor(
    1.5
)

print(
    "Finite:",
    torch.isfinite(
        example_loss
    ).item()
)


# 66. Check Loss During Training

A useful defensive check:

```python
if not torch.isfinite(loss):
    ...
```

If the loss becomes non-finite, stop and debug instead of continuing blindly.


In [ ]:
def assert_finite_loss(
    loss
):
    if not torch.isfinite(
        loss
    ):
        raise RuntimeError(
            "Loss became non-finite."
        )

example_loss = torch.tensor(
    0.7
)

assert_finite_loss(
    example_loss
)

print(
    "Loss is finite."
)


# 67. Check Gradient Finiteness


In [ ]:
def gradients_are_finite(
    model
):
    for parameter in model.parameters():
        if parameter.grad is not None:
            if not torch.isfinite(
                parameter.grad
            ).all():
                return False

    return True

print(
    "Gradients finite:",
    gradients_are_finite(
        model
    )
)


# 68. Stable Training Habit — Use Reproducible Seeds

A fixed seed can help reproduce:

- Weight initialization
- Data splitting
- DataLoader shuffling
- Random augmentation behavior


In [ ]:
torch.manual_seed(
    42
)

print(
    torch.randn(
        3
    )
)


# 69. Stable Training Habit — Keep the Best Checkpoint

Do not assume the final epoch is the best epoch.

Save the model when the validation metric improves.

This protects against:

- Later overfitting
- Instability
- Metric degradation


# 70. Stable Training Habit — Change One Variable at a Time

If you simultaneously change:

- Architecture
- Learning rate
- Augmentation
- Weight decay
- Dropout
- Batch size

and performance changes, you do not know why.

Controlled experiments are easier to interpret.


# 71. Creating a Small Comparison Dataset

Now we will compare several regularization strategies on the same synthetic classification problem.

We will deliberately create:

- A relatively small training set
- A larger model

to make overfitting easier to observe.


In [ ]:
torch.manual_seed(
    123
)

num_train = 120
num_val = 600

train_x = torch.randn(
    num_train,
    20
)

val_x = torch.randn(
    num_val,
    20
)

true_weight = torch.randn(
    20,
    3
)

train_logits_true = (
    train_x
    @ true_weight
)

val_logits_true = (
    val_x
    @ true_weight
)

train_y = train_logits_true.argmax(
    dim=1
)

val_y = val_logits_true.argmax(
    dim=1
)

# Introduce some training-label noise.
noise_indices = torch.randperm(
    num_train
)[:20]

train_y[
    noise_indices
] = torch.randint(
    0,
    3,
    (20,)
)

print(
    "Train:",
    train_x.shape,
    train_y.shape
)

print(
    "Validation:",
    val_x.shape,
    val_y.shape
)


# 72. DataLoaders for the Comparison


In [ ]:
train_dataset = TensorDataset(
    train_x,
    train_y
)

val_dataset = TensorDataset(
    val_x,
    val_y
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)


# 73. Comparison Model

We will use the same architecture for every experiment.

The only differences will be regularization settings.


In [ ]:
class ComparisonMLP(nn.Module):
    def __init__(
        self,
        dropout_p=0.0
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                20,
                128
            ),
            nn.ReLU(),
            nn.Dropout(
                p=dropout_p
            ),

            nn.Linear(
                128,
                128
            ),
            nn.ReLU(),
            nn.Dropout(
                p=dropout_p
            ),

            nn.Linear(
                128,
                3
            )
        )

    def forward(self, x):
        return self.network(
            x
        )


# 74. Reusable Training Function


In [ ]:
def run_training_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    loss_sum = 0.0
    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            inputs
        )

        loss = criterion(
            logits,
            targets
        )

        assert_finite_loss(
            loss
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        loss_sum += (
            loss.item()
            * batch_size
        )

        correct += (
            logits.argmax(
                dim=1
            )
            == targets
        ).sum().item()

        total += batch_size

    return (
        loss_sum / total,
        correct / total
    )


# 75. Reusable Evaluation Function


In [ ]:
def run_evaluation_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    loss_sum = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                inputs
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            loss_sum += (
                loss.item()
                * batch_size
            )

            correct += (
                logits.argmax(
                    dim=1
                )
                == targets
            ).sum().item()

            total += batch_size

    return (
        loss_sum / total,
        correct / total
    )


# 76. Experiment Function

We will compare:

1. No explicit regularization
2. Dropout
3. Weight decay
4. Dropout + weight decay


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

def train_regularization_experiment(
    dropout_p=0.0,
    weight_decay=0.0,
    epochs=60,
    seed=42
):
    torch.manual_seed(
        seed
    )

    model = ComparisonMLP(
        dropout_p=dropout_p
    ).to(
        device
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=weight_decay
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_loss = float(
        "inf"
    )

    best_state = copy.deepcopy(
        model.state_dict()
    )

    for _ in range(
        epochs
    ):
        train_loss, train_acc = (
            run_training_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                device
            )
        )

        val_loss, val_acc = (
            run_evaluation_epoch(
                model,
                val_loader,
                criterion,
                device
            )
        )

        history[
            "train_loss"
        ].append(
            train_loss
        )

        history[
            "train_acc"
        ].append(
            train_acc
        )

        history[
            "val_loss"
        ].append(
            val_loss
        )

        history[
            "val_acc"
        ].append(
            val_acc
        )

        if val_loss < best_val_loss:
            best_val_loss = (
                val_loss
            )

            best_state = copy.deepcopy(
                model.state_dict()
            )

    model.load_state_dict(
        best_state
    )

    return (
        model,
        history
    )


# 77. Running the Regularization Experiments

These are small educational experiments.

The exact curves may vary slightly across hardware and PyTorch versions.


In [ ]:
baseline_model, baseline_history = (
    train_regularization_experiment(
        dropout_p=0.0,
        weight_decay=0.0
    )
)

dropout_model, dropout_history = (
    train_regularization_experiment(
        dropout_p=0.3,
        weight_decay=0.0
    )
)

decay_model, decay_history = (
    train_regularization_experiment(
        dropout_p=0.0,
        weight_decay=1e-3
    )
)

combined_model, combined_history = (
    train_regularization_experiment(
        dropout_p=0.3,
        weight_decay=1e-3
    )
)

print(
    "Experiments complete."
)


# 78. Comparing Validation Loss


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    baseline_history[
        "val_loss"
    ],
    label="No regularization"
)

plt.plot(
    dropout_history[
        "val_loss"
    ],
    label="Dropout"
)

plt.plot(
    decay_history[
        "val_loss"
    ],
    label="Weight decay"
)

plt.plot(
    combined_history[
        "val_loss"
    ],
    label="Dropout + weight decay"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Regularization Strategy Comparison")
plt.legend()
plt.show()


# 79. Comparing Validation Accuracy


In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    baseline_history[
        "val_acc"
    ],
    label="No regularization"
)

plt.plot(
    dropout_history[
        "val_acc"
    ],
    label="Dropout"
)

plt.plot(
    decay_history[
        "val_acc"
    ],
    label="Weight decay"
)

plt.plot(
    combined_history[
        "val_acc"
    ],
    label="Dropout + weight decay"
)

plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Validation Accuracy Comparison")
plt.legend()
plt.show()


# 80. Compare Best Validation Performance


In [ ]:
def summarize_history(
    name,
    history
):
    best_loss = min(
        history[
            "val_loss"
        ]
    )

    best_acc = max(
        history[
            "val_acc"
        ]
    )

    final_train_acc = history[
        "train_acc"
    ][-1]

    print(
        f"{name:24s} | "
        f"best val loss = {best_loss:.4f} | "
        f"best val acc = {best_acc:.3f} | "
        f"final train acc = {final_train_acc:.3f}"
    )

summarize_history(
    "No regularization",
    baseline_history
)

summarize_history(
    "Dropout",
    dropout_history
)

summarize_history(
    "Weight decay",
    decay_history
)

summarize_history(
    "Dropout + decay",
    combined_history
)


# 81. Important Interpretation Rule

Do not expect one regularization strategy to always win.

The result depends on:

- Dataset
- Model size
- Noise
- Learning rate
- Number of epochs
- Regularization strength

The purpose of this comparison is to learn the experimental process:

> Keep the pipeline fixed and change one regularization choice at a time.


# 82. Too Much Regularization Can Cause Underfitting

Suppose we use:

$$
dropout=0.8
$$

and very large weight decay.

The network may become so constrained that it cannot fit even the training set.

Signs:

- Training loss stays high
- Training accuracy stays low
- Validation is also poor

Regularization should not destroy useful model capacity.


# 83. Regularization Strength Is a Hyperparameter

Examples:

- Dropout probability
- Weight decay coefficient
- L1 coefficient
- Augmentation intensity
- Early-stopping patience

These should be selected using validation data.

Do not tune them on the test set.


# 84. Initialization Debugging Checklist

If a custom network trains poorly from the first step, inspect:

1. Weight mean
2. Weight standard deviation
3. Activation distributions
4. Gradient norms
5. Whether weights are accidentally all identical
6. Whether initialization matches the activation function


In [ ]:
model = ComparisonMLP(
    dropout_p=0.0
)

for name, parameter in model.named_parameters():
    if (
        "weight"
        in name
    ):
        print(
            name,
            "| mean:",
            parameter.mean().item(),
            "| std:",
            parameter.std().item()
        )


# 85. Stable Training Debugging Checklist

If training becomes unstable:

1. Check for `nan` or `inf`
2. Reduce learning rate
3. Inspect gradient norms
4. Inspect activation scale
5. Check input normalization
6. Check loss/target compatibility
7. Check initialization
8. Consider gradient clipping
9. Check data for corrupted values
10. Check mixed-precision settings if used
11. Verify regularization is not excessive
12. Verify optimizer configuration


# 86. Common Mistake — Using Dropout During Evaluation

If you forget:

```python
model.eval()
```

dropout remains active during validation.

Predictions become unnecessarily random.

Always switch modes correctly.


# 87. Common Mistake — Too Much Dropout

Very high dropout can remove too much useful signal.

A model that should fit the training data may instead underfit badly.

More regularization is not automatically better.


# 88. Common Mistake — Weight Decay on Everything Without Thought

Some advanced architectures treat certain parameters differently.

For example, some training setups exclude:

- Biases
- Normalization parameters

from weight decay.

For beginner models, applying one weight-decay value to all trainable parameters is a reasonable starting point.

Later, parameter groups can provide finer control.


# 89. Common Mistake — Xavier Everywhere

Xavier is not automatically the best initialization for every activation.

For ReLU-heavy hidden layers, Kaiming initialization is often more appropriate.

Initialization should match the network's nonlinearities.


# 90. Common Mistake — Kaiming Everywhere

The reverse is also true.

Kaiming initialization is designed around rectifier-style nonlinearities.

The best initialization for the final layer or for tanh networks may differ.

Avoid blindly applying one rule everywhere.


# 91. Common Mistake — Clipping Gradients Before `backward()`

Gradient clipping requires gradients to already exist.

Correct order:

```python
optimizer.zero_grad()

loss.backward()

clip_grad_norm_(
    model.parameters(),
    ...
)

optimizer.step()
```


# 92. Common Mistake — Clipping After `optimizer.step()`

If clipping happens after the optimizer step, it is too late for that update.

Clip before:

```python
optimizer.step()
```


# 93. Common Mistake — Ignoring Validation Curves

Training loss alone cannot tell you whether regularization is working.

Always compare training and validation behavior.


# 94. Common Mistake — Changing Many Things at Once

If you simultaneously change:

- Initialization
- Dropout
- Weight decay
- Learning rate
- Architecture

you lose the ability to understand which change caused the result.

Use controlled ablations.


# 95. A Clean Stable Training Skeleton


In [ ]:
def stable_training_step(
    model,
    inputs,
    targets,
    criterion,
    optimizer,
    max_grad_norm=None
):
    model.train()

    optimizer.zero_grad()

    outputs = model(
        inputs
    )

    loss = criterion(
        outputs,
        targets
    )

    assert_finite_loss(
        loss
    )

    loss.backward()

    if max_grad_norm is not None:
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=max_grad_norm
        )

    optimizer.step()

    return loss.item()


# 96. Why This Skeleton Is Useful

It makes the important order explicit:

$$
\boxed{
zero\_grad
\rightarrow
forward
\rightarrow
loss
\rightarrow
finite\ check
\rightarrow
backward
\rightarrow
clip
\rightarrow
step
}
$$

The exact training system may become more complex later, but this order is a strong baseline.


# 97. Practice Exercises

Try these before looking at the solutions.

## Exercise 1

Explain the difference between underfitting and overfitting.

## Exercise 2

Create:

```python
nn.Dropout(
    p=0.4
)
```

and compare outputs in training and evaluation modes.

## Exercise 3

Create an optimizer using:

`AdamW`

with:

$$
weight\_decay=10^{-4}
$$

## Exercise 4

Calculate manually:

$$
L1
$$

and:

$$
L2
$$

penalties for:

$$
\begin{array}{|c|c|c|}
\hline
2 & -1 & 3 \\
\hline
\end{array}
$$

## Exercise 5

Apply Xavier initialization to an `nn.Linear` layer.

## Exercise 6

Apply Kaiming initialization to a ReLU network.

## Exercise 7

Create a deep network with intentionally huge initialization and inspect activation standard deviations.

## Exercise 8

Compute gradient norms after a backward pass.

## Exercise 9

Clip gradient norm to:

$$
1.0
$$

## Exercise 10

Build a training loop with:

- Dropout
- Weight decay
- Best checkpointing
- Early stopping


# 98. Conceptual Challenges

Answer without running code first.

## Challenge 1

Why can a model with lower training accuracy generalize better?

## Challenge 2

Why is dropout disabled during evaluation?

## Challenge 3

What is the main difference in intuition between L1 and L2 regularization?

## Challenge 4

Why is data augmentation considered regularization?

## Challenge 5

Why can all-zero hidden-layer weights prevent useful learning?

## Challenge 6

Why is Kaiming initialization commonly paired with ReLU?

## Challenge 7

What are vanishing gradients?

## Challenge 8

What are exploding gradients?

## Challenge 9

Why must gradient clipping happen after backward but before optimizer step?

## Challenge 10

Why can excessive regularization cause underfitting?


# 99. Exercise Solutions


In [ ]:
# Exercise 2
drop = nn.Dropout(
    p=0.4
)

x = torch.ones(
    10
)

drop.train()

print(
    "Exercise 2 train:",
    drop(x)
)

drop.eval()

print(
    "Exercise 2 eval:",
    drop(x)
)

# Exercise 3
exercise_model = nn.Linear(
    5,
    2
)

exercise_optimizer = (
    torch.optim.AdamW(
        exercise_model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
)

print(
    "Exercise 3:",
    exercise_optimizer
)

# Exercise 4
exercise_weights = torch.tensor([
    2.0,
    -1.0,
    3.0
])

exercise_l1 = (
    exercise_weights.abs().sum()
)

exercise_l2 = (
    exercise_weights.pow(2).sum()
)

print(
    "Exercise 4 L1:",
    exercise_l1.item()
)

print(
    "Exercise 4 L2:",
    exercise_l2.item()
)

# Exercise 5
xavier_layer = nn.Linear(
    20,
    10
)

nn.init.xavier_uniform_(
    xavier_layer.weight
)

nn.init.zeros_(
    xavier_layer.bias
)

print(
    "Exercise 5 std:",
    xavier_layer.weight.std().item()
)

# Exercise 6
kaiming_model = nn.Sequential(
    nn.Linear(
        20,
        64
    ),
    nn.ReLU(),
    nn.Linear(
        64,
        3
    )
)

nn.init.kaiming_normal_(
    kaiming_model[0].weight,
    nonlinearity="relu"
)

nn.init.zeros_(
    kaiming_model[0].bias
)

print(
    "Exercise 6 complete."
)

# Exercise 8 and 9
inputs = torch.randn(
    16,
    20
)

targets = torch.randint(
    0,
    3,
    (16,)
)

criterion = nn.CrossEntropyLoss()

logits = kaiming_model(
    inputs
)

loss = criterion(
    logits,
    targets
)

loss.backward()

for name, parameter in (
    kaiming_model.named_parameters()
):
    if parameter.grad is not None:
        print(
            "Exercise 8",
            name,
            parameter.grad.norm().item()
        )

norm_before_clip = (
    torch.nn.utils.clip_grad_norm_(
        kaiming_model.parameters(),
        max_norm=1.0
    )
)

print(
    "Exercise 9 norm before clip:",
    norm_before_clip.item()
)


# 100. Key Takeaways

In this notebook, we learned:

- What regularization means
- Underfitting
- Overfitting
- Dropout
- Train vs evaluation dropout behavior
- Weight decay
- L1 regularization
- L2 regularization
- Adam vs AdamW intuition
- Data augmentation as regularization
- Early stopping
- Why initialization matters
- Symmetry problems
- PyTorch default initialization
- Fan-in and fan-out
- Xavier initialization
- Kaiming initialization
- Activation scale
- Vanishing activations
- Exploding activations
- Vanishing gradients
- Exploding gradients
- Gradient norms
- Gradient clipping
- Stable training habits
- Comparing regularization strategies
- Common stability mistakes

A useful mental model is:

$$
\boxed{
\text{Good Generalization}
=
\text{Enough Capacity}
+
\text{Appropriate Regularization}
+
\text{Stable Optimization}
}
$$

And stable optimization depends on:

$$
\boxed{
\text{Input Scale}
+
\text{Initialization}
+
\text{Learning Rate}
+
\text{Gradient Scale}
}
$$


# 101. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What is regularization?
2. What is underfitting?
3. What is overfitting?
4. What does dropout do during training?
5. What happens to dropout during evaluation?
6. What does weight decay try to encourage?
7. What is L1 regularization?
8. What is L2 regularization?
9. Why can L1 encourage sparsity?
10. Why is data augmentation a form of regularization?
11. What is early stopping?
12. Why should the best checkpoint still be saved when using early stopping?
13. Why does initialization matter?
14. Why are all-zero hidden-layer weights problematic?
15. What are fan-in and fan-out?
16. When is Xavier initialization commonly used?
17. When is Kaiming initialization commonly used?
18. What are vanishing activations?
19. What are exploding activations?
20. What are vanishing gradients?
21. What are exploding gradients?
22. What does gradient clipping do?
23. When should gradient clipping occur in the training step?
24. Why can too much regularization hurt?
25. Why should regularization choices be tuned on validation data rather than test data?


# Next Notebook

# 18 — Debugging, Reproducibility, and PyTorch Best Practices

In the next notebook, we will study:

- A systematic PyTorch debugging workflow
- Shape debugging
- Dtype debugging
- Device debugging
- Gradient debugging
- Detecting NaNs and infinities
- Reproducibility
- Random seeds
- Deterministic behavior
- Saving experiment configuration
- Model summaries
- Parameter inspection
- Data leakage checks
- Sanity-check experiments
- Overfitting one tiny batch
- Clean project organization
- Practical PyTorch best practices
